In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será salva
database_name = "fato"

# Nome da tabela de exportações NCM
table_name = "ft_exportacoes_ncm"

# Caminho alvo no formato <database>.<table> para salvar a tabela no Hive/Delta
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "SK_EXPORTACAO_UF"

In [0]:
# Bases utilizadas no relacionamento exportacao por municipio
# Caminho para a base de exportação consolidada (nível município)
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_CONSOLIDADA/"

# Caminho para a base de Unidades Federativas (UF)
silver_path_uf = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/UF/"

In [0]:
# Leitura dos dados de exportação por município a partir do caminho Delta Lake (silver layer)
df_exp_uf = spark.read.format("delta").load(silver_path_s)

# Leitura dos dados de unidades federativas (UF) a partir do caminho Delta Lake (silver layer)
df_uf = spark.read.format("delta").load(silver_path_uf)

In [0]:
df_exp_uf.createOrReplaceTempView("df_exp_uf")

df_uf.createOrReplaceTempView("df_uf")

In [0]:
# Query para seleção de colunas relevantes da tabela de exportações consolidadas por UF.
# As colunas selecionadas incluem informações temporais, de produto (NCM), unidade, país, unidade de despacho,
# quantidade estatística, peso líquido e valor FOB.
# O relacionamento com tabelas geográficas pode ser realizado posteriormente utilizando as colunas CO_PAIS e CO_URF.
# Esta query serve como base para análises e integrações futuras.

query = """
select
    concat(e.CO_ANO,'-',e.CO_MES) as ano_mes,
    e.CO_NCM,
    e.CO_UNID,

    --relacionamento com tabelas geograficas:
    e.CO_PAIS,
    e.CO_URF,
    e.QT_ESTAT,

    e.KG_LIQUIDO,
    e.VL_FOB

from df_exp_uf e 
"""

In [0]:
# Executa a query SQL definida anteriormente e armazena o resultado em um DataFrame Spark.
# Esta consulta seleciona e transforma dados da tabela temporária 'df_exp_uf',
# incluindo colunas de data, NCM, unidade, país, unidade de despacho, quantidade estatística,
# peso líquido e valor FOB, para posterior processamento ou análise.
df_join = spark.sql(query)

In [0]:
df_join.count()

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS database_name")

In [0]:
save_hive_table(df_join, target_path, pk)